## 2 — GLM-MRP (county-level estimates)
Frequentist logistic regression with **all fixed effects** + MRP poststratification at the county level.

Model formula (no random effects):
```
happening_bin ~ C(gender) + C(race4) + C(educ_category)
              + co2_per_capita + dem_share_two_party
              + pct_drive_alone + pct_samesex_hh
```
All four geographic covariates are **county-level** (joined onto each respondent
via `county_fips`). State / region appear nowhere here.

Output: `outputs/estimates/glm_mrp_county_estimates.csv`

In [5]:
%pip install statsmodels -q

Note: you may need to restart the kernel to use updated packages.


In [6]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

DATA_DIR   = Path("../test_data/processed/")
OUTPUT_DIR = Path("../outputs/")
OUTCOME    = "happening_bin"
MODEL_NAME = "glm_mrp"
STATE_CSV_NAME = "glm_mrp_state_estimates.csv"

FORMULA = (
    "happening_bin ~ C(gender) + C(race4) + C(educ_category)"
    " + co2_per_capita + dem_share_two_party + pct_drive_alone + pct_samesex_hh"
)

STATE_NAMES = {
    "01":"Alabama","02":"Alaska","04":"Arizona","05":"Arkansas","06":"California",
    "08":"Colorado","09":"Connecticut","10":"Delaware","11":"District of Columbia",
    "12":"Florida","13":"Georgia","15":"Hawaii","16":"Idaho","17":"Illinois",
    "18":"Indiana","19":"Iowa","20":"Kansas","21":"Kentucky","22":"Louisiana",
    "23":"Maine","24":"Maryland","25":"Massachusetts","26":"Michigan",
    "27":"Minnesota","28":"Mississippi","29":"Missouri","30":"Montana",
    "31":"Nebraska","32":"Nevada","33":"New Hampshire","34":"New Jersey",
    "35":"New Mexico","36":"New York","37":"North Carolina","38":"North Dakota",
    "39":"Ohio","40":"Oklahoma","41":"Oregon","42":"Pennsylvania",
    "44":"Rhode Island","45":"South Carolina","46":"South Dakota",
    "47":"Tennessee","48":"Texas","49":"Utah","50":"Vermont",
    "51":"Virginia","53":"Washington","54":"West Virginia","55":"Wisconsin",
    "56":"Wyoming",
}

In [7]:
# ── Build county-level covariate table ─────────────────────────────────────
# poststrat_county already carries co2_per_capita and dem_share_two_party at the
# county level; merge in pct_drive_alone, pct_samesex_hh from the ACS extras.
ps_county = pd.read_csv(DATA_DIR / "poststrat_county.csv",
                         dtype={"county_fips": str, "state_fips": str})
acs_county = pd.read_csv(DATA_DIR / "acs_county_extra_covariates.csv",
                          dtype={"county_fips": str, "state_fips": str})

ps_county = ps_county.merge(
    acs_county[["county_fips", "pct_drive_alone", "pct_samesex_hh"]],
    on="county_fips", how="left",
)

county_cov = (
    ps_county[["county_fips", "state_fips",
               "co2_per_capita", "dem_share_two_party",
               "pct_drive_alone", "pct_samesex_hh"]]
    .drop_duplicates(subset=["county_fips"])
    .reset_index(drop=True)
)
# Impute any missing covariate values with the national mean
for c in ["co2_per_capita", "dem_share_two_party", "pct_drive_alone", "pct_samesex_hh"]:
    county_cov[c] = county_cov[c].fillna(county_cov[c].mean())

print(f"poststrat_county rows: {len(ps_county):,}  | counties: {ps_county['county_fips'].nunique():,}")
print(f"county covariate table: {county_cov.shape}")
print(county_cov.head(4).to_string(index=False))

poststrat_county rows: 99,940  | counties: 3,143
county covariate table: (3143, 6)
county_fips state_fips  co2_per_capita  dem_share_two_party  pct_drive_alone  pct_samesex_hh
      01001         01       81.134604             0.266411         0.843044        0.005769
      01003         01       10.888320             0.206524         0.794437        0.005769
      01005         01       12.489276             0.425850         0.832330        0.005769
      01007         01       10.493996             0.176151         0.848439        0.005769


In [8]:
# ── Load survey + merge county covariates ─────────────────────────────────
survey = pd.read_csv(DATA_DIR / "climate_survey_responses_recoded.csv",
                     dtype={"state_fips": str, "county_fips": str})
survey = survey.dropna(subset=[OUTCOME]).copy()
survey[OUTCOME] = survey[OUTCOME].astype(float)
survey["educ_category"] = survey["educ_category"].astype(str)
survey = survey.merge(county_cov.drop(columns=["state_fips"]),
                       on="county_fips", how="left")

print(f"Survey rows: {len(survey):,}  ({survey[OUTCOME].mean()*100:.1f}% Yes)")
print(f"Missing co2 (county join): {survey['co2_per_capita'].isna().sum()}")
print(f"Missing drive_alone: {survey['pct_drive_alone'].isna().sum()}")

Survey rows: 1,011  (57.9% Yes)
Missing co2 (county join): 2
Missing drive_alone: 2


In [9]:
# ── Build county-level poststrat frame with merged covariates ─────────────
ps = ps_county[["county_fips", "state_fips", "gender", "race4", "educ_category",
                  "N_rounded", "co2_per_capita", "dem_share_two_party",
                  "pct_drive_alone", "pct_samesex_hh"]].copy()
ps["educ_category"] = ps["educ_category"].astype(str)
# Backfill any cells whose county-level covariates are NaN with the national mean
for c in ["co2_per_capita", "dem_share_two_party", "pct_drive_alone", "pct_samesex_hh"]:
    ps[c] = ps[c].fillna(county_cov[c].mean())

print(f"Poststrat rows: {len(ps):,}  |  counties: {ps['county_fips'].nunique():,}")

Poststrat rows: 99,940  |  counties: 3,143


In [10]:
# ── Fit GLM ───────────────────────────────────────────────────────────────
caught = []
with warnings.catch_warnings(record=True) as _w:
    warnings.simplefilter("always")
    model = smf.glm(
        formula=FORMULA, data=survey,
        family=sm.families.Binomial(link=sm.families.links.Logit()),
    ).fit()
    caught = list(_w)

pseudo_r2 = 1 - model.llf / model.llnull
print(f"AIC: {model.aic:.1f}  |  Pseudo R² (McFadden): {pseudo_r2:.4f}")
print(f"Convergence warnings: {len(caught)}")
print(model.summary().tables[1])

AIC: 1365.2  |  Pseudo R² (McFadden): 0.0236
Convergence warnings: 0
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                -1.3372      1.174     -1.139      0.255      -3.638       0.963
C(gender)[T.Male]         0.0761      0.131      0.582      0.561      -0.180       0.333
C(race4)[T.Hispanic]     -0.3032      0.272     -1.115      0.265      -0.837       0.230
C(race4)[T.Other]        -0.1492      0.339     -0.440      0.660      -0.814       0.516
C(race4)[T.White]        -0.3868      0.242     -1.600      0.110      -0.861       0.087
C(educ_category)[T.2]     0.2616      0.220      1.190      0.234      -0.169       0.692
C(educ_category)[T.3]     0.2457      0.217      1.131      0.258      -0.180       0.672
C(educ_category)[T.4]     0.5665      0.222      2.555      0.011       0.132       1.001
co2_per_capita           -0.005

In [11]:
# ── Predict + poststratify by county ──────────────────────────────────────
ps["predicted_prob"] = model.predict(ps)

estimates = (
    ps.groupby(["county_fips", "state_fips"])
    .apply(lambda g: np.average(g["predicted_prob"], weights=g["N_rounded"]),
           include_groups=False)
    .reset_index(name="happening_estimate")
)
estimates["state_name"] = estimates["state_fips"].map(STATE_NAMES)
estimates = estimates[["county_fips", "state_fips", "state_name", "happening_estimate"]]
print(f"County estimates: {len(estimates):,}  (counties)")
print(f"Range: [{estimates['happening_estimate'].min():.4f}, "
      f"{estimates['happening_estimate'].max():.4f}]")
print(f"Mean: {estimates['happening_estimate'].mean():.4f}")

County estimates: 3,143  (counties)
Range: [0.0000, 0.9597]
Mean: 0.4788


In [12]:
est_dir  = OUTPUT_DIR / "estimates"
diag_dir = OUTPUT_DIR / "diagnostics"
est_dir.mkdir(parents=True, exist_ok=True)
diag_dir.mkdir(parents=True, exist_ok=True)

out_path = est_dir / f"{MODEL_NAME}_county_estimates.csv"
estimates.to_csv(out_path, index=False)
print(f"Saved → {out_path}  (rows: {len(estimates):,})")

diag_path = diag_dir / f"{MODEL_NAME}_county_summary.txt"
with open(diag_path, "w") as f:
    f.write("GLM-MRP County — Diagnostic Summary\n" + "=" * 55 + "\n\n")
    f.write(f"Formula: {FORMULA}\n")
    f.write(f"Observations: {int(model.nobs)}\n")
    f.write(f"AIC: {model.aic:.2f}\n")
    f.write(f"Pseudo R² (McFadden): {pseudo_r2:.4f}\n\n")
    f.write(str(model.summary()) + "\n\n")
    f.write(f"County estimates — mean:{estimates['happening_estimate'].mean():.4f}  "
            f"min:{estimates['happening_estimate'].min():.4f}  "
            f"max:{estimates['happening_estimate'].max():.4f}\n")
print(f"Diagnostics → {diag_path}")

Saved → ../outputs/estimates/glm_mrp_county_estimates.csv  (rows: 3,143)
Diagnostics → ../outputs/diagnostics/glm_mrp_county_summary.txt


In [13]:
# ── State-level rollup diagnostic ──────────────────────────────────────────
# Roll county estimates up to state via population-weighted average, then compare
# with the same model's state-level CSV (from the parallel state notebook).
state_rollup = (
    estimates.merge(
        ps_county.groupby("county_fips")["N_rounded"].sum().reset_index(name="county_pop"),
        on="county_fips", how="left",
    )
    .dropna(subset=["happening_estimate"])
    .groupby("state_fips")
    .apply(lambda g: np.average(g["happening_estimate"], weights=g["county_pop"]),
           include_groups=False)
    .reset_index(name="county_rollup")
)
state_rollup["state_name"] = state_rollup["state_fips"].map(STATE_NAMES)

state_csv = OUTPUT_DIR / "estimates" / f"{STATE_CSV_NAME}"
if state_csv.exists():
    state_est = pd.read_csv(state_csv, dtype={"state_fips": str})
    cmp = state_rollup.merge(state_est[["state_fips", "estimate"]], on="state_fips", how="left")
    cmp["abs_diff"] = (cmp["county_rollup"] - cmp["estimate"]).abs()
    print(f"\nState rollup vs {STATE_CSV_NAME}:")
    print(f"  mean |diff|: {cmp['abs_diff'].mean():.4f}")
    print(f"  max  |diff|: {cmp['abs_diff'].max():.4f}")
    print(cmp[["state_fips", "state_name", "county_rollup", "estimate", "abs_diff"]]
          .sort_values("abs_diff", ascending=False).head(10).to_string(index=False))
else:
    print(f"\nNo state CSV found at {state_csv} — skipping rollup comparison.")


State rollup vs glm_mrp_state_estimates.csv:
  mean |diff|: 0.0684
  max  |diff|: 0.1982
state_fips    state_name  county_rollup  estimate  abs_diff
        09   Connecticut       0.515117  0.713290  0.198173
        16         Idaho       0.424783  0.256929  0.167855
        24      Maryland       0.631851  0.799705  0.167854
        40      Oklahoma       0.486329  0.329332  0.156997
        47     Tennessee       0.496474  0.347923  0.148551
        54 West Virginia       0.418901  0.272844  0.146057
        21      Kentucky       0.500487  0.354854  0.145633
        49          Utah       0.552212  0.411884  0.140327
        12       Florida       0.636590  0.518796  0.117793
        05      Arkansas       0.446034  0.337219  0.108815
